# 04. PSNR / SSIM 평가

한국 음식 test set 위에서 세 모델의 colorization 품질을 비교합니다.

* PSNR: 원본 컬러와 예측 사이의 신호-잡음 비.
* SSIM: 구조적 유사도 (luminance / contrast / structure 결합).

PSNR/SSIM 은 colorization 평가의 절대 지표는 아닙니다 (perceptual quality 와 항상 비례하지 않음). 정성 비교와 함께 봐야 합니다.

In [ ]:
import os, glob
import numpy as np
import cv2
import torch
from PIL import Image
from math import log10, sqrt
from tqdm import tqdm

## 1. PSNR / SSIM 함수

In [ ]:
def psnr(orig, pred):
    mse = np.mean((orig.astype(np.float32) - pred.astype(np.float32)) ** 2)
    if mse == 0:
        return 100.0
    return 20 * log10(255.0 / sqrt(mse))

from skimage.metrics import structural_similarity as ssim
def ssim_score(orig, pred):
    return ssim(orig, pred, channel_axis=-1, data_range=255)

## 2. 모델별 예측 폴더

각 파이프라인 노트북이 만든 출력 폴더 경로.

In [ ]:
MODELS = {
    'ChromaGAN':       'ChromaGAN_run/Output_10/test',
    'DeOldify':        'DeOldify_run/Output_korean_food',
    'InstColorization':'InstColor_run/Output_korean_food',
}
GT_DIR = '../DATASET/test/color_image'

## 3. 비교 평가

In [ ]:
def evaluate(pred_dir):
    psnrs, ssims = [], []
    for gt_path in tqdm(sorted(glob.glob(os.path.join(GT_DIR, '*.png')))):
        base = os.path.basename(gt_path).replace('_color.png', '')
        cand = glob.glob(os.path.join(pred_dir, f'{base}*.png'))
        if not cand:
            continue
        gt = np.array(Image.open(gt_path).convert('RGB'))
        pr = np.array(Image.open(cand[0]).convert('RGB').resize(gt.shape[1::-1]))
        psnrs.append(psnr(gt, pr))
        ssims.append(ssim_score(gt, pr))
    return (np.mean(psnrs) if psnrs else None,
            np.mean(ssims) if ssims else None,
            len(psnrs))

rows = []
for name, d in MODELS.items():
    if os.path.isdir(d):
        p, s, n = evaluate(d)
    else:
        p, s, n = (None, None, 0)
    rows.append((name, p, s, n))
import pandas as pd
df = pd.DataFrame(rows, columns=['model', 'PSNR', 'SSIM', 'N'])
df

## 메모

* DeOldify / InstColorization 은 보고서 시점 기준 학습이 끝까지 가지 못해, 위 표는 사전학습 weight 기반 inference 만으로 산출됩니다.
* ChromaGAN 만이 한국 음식 fine-tune 결과로 평가됩니다.